## Agent Concepts & Execution Control

This notebook provides a hands-on introduction to building, managing, and controlling AI Agents using LangChain and Google's Gemini models. It serves as a practical foundation for understanding how agents reason and interact with tools before migrating to more advanced, graph-based routing (like LangGraph).

### **Key Concepts Explored:**
- **Defining Custom Tools:** Creating modular Python functions equipped with the `@tool` decorator to grant the LLM specific capabilities (e.g., math calculations or fetching local prices).
- **Agent Architectures:** Initializing different models of reasoning, including structured tool-calling agents and ReAct (Reason + Act) agents.
- **Handling Parsing Errors:** Implementing graceful error recovery using `handle_parsing_errors` to catch and correct situations where the LLM produces invalid formatting or hallucinations.
- **Tracing Intermediate Steps:** Peeking into the agent's action-observation loop (`return_intermediate_steps`) to see exactly how it arrives at its final answer.
- **Execution Safeguards:** Applying hard constraints on agents using `max_iterations` and `max_execution_time` (timeout bounds) to prevent infinite loops and runaway execution.

---
*Note: This notebook relies on `langchain`, `langchain_google_genai`, and integrations like `WikipediaAPIWrapper` to empower the agents with live search reasoning.*


In [ ]:
!pip install langgraph langchain_google_genai   ## Install the necessary libraries

In [ ]:

from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import tool
import os
from google.colab import userdata, files
from langchain_google_genai import ChatGoogleGenerativeAI


In [ ]:
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:

llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.1,
        google_api_key=os.environ.get("GEMINI_API_KEY")
    )

In [ ]:
@tool
def get_food_price(item_name: str) -> float:
    """Returns the current price of a specific food item."""
    prices = {
        "pizza": 15.50,
        "burger": 8.75,
        "soda": 2.50,
        "fries": 4.00,
    }
    return prices.get(item_name.lower(), 0.0)

@tool
def calculator(expression: str) -> float:
    """Performs a mathematical calculation."""
    return eval(expression)

In [ ]:
from langchain.agents import create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain import hub

tools = [get_food_price, calculator]

prompt = hub.pull("hwchase17/openai-functions-agent")

agent = create_openai_tools_agent(llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
question = "What is the average price of a pizza and a burger?"

for step in agent_executor.iter({"input": question}):
    if "intermediate_step" in step:
        action, observation = step["intermediate_step"][0]

    elif "output" in step:
        print(f"FINAL ANSWER: {step['output']}")
        break



> Entering new None chain...

Invoking: `get_food_price` with `{'item_name': 'pizza'}`


15.5
Invoking: `get_food_price` with `{'item_name': 'burger'}`


8.75
Invoking: `calculator` with `{'expression': '(15.5 + 8.75) / 2'}`


12.125The average price of a pizza and a burger is 12.125.

> Finished chain.
FINAL ANSWER: The average price of a pizza and a burger is 12.125.


## Returning structured output

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

class FinalAnswer(BaseModel):
    final_answer: str = Field(description="The final, human-readable answer to the user's question.")
    sources_used: List[str] = Field(description="A list of tools or sources used to formulate the answer.")

@tool
def get_food_price(item_name: str) -> float:
    """Returns the current price of a specific food item."""
    prices = {"pizza": 15.50, "burger": 8.75, "soda": 2.50}
    return prices.get(item_name.lower(), 0.0)

@tool
def calculator(expression: str) -> float:
    """Performs a mathematical calculation."""
    return eval(expression)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    temperature=0.1,
    api_key=os.environ.get("GEMINI_API_KEY")
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI that can use tools to answer user questions."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

tools = [get_food_price, calculator]

agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

question = "What is 10+5?"

response = agent_executor.invoke({"input": question},return_only_outputs=True,)
print(response["output"])




> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'expression': '10+5'}`


1510 + 5 = 15


> Finished chain.
10 + 5 = 15



## Handling parsing errors

In [ ]:
!pip install langchain_community wikipedia

In [ ]:
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_google_genai import ChatGoogleGenerativeAI

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)
tools = [tool]

prompt = hub.pull("hwchase17/react")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    temperature=0.1,
    api_key=os.getenv("GEMENI_API_KEY")
)


agent = create_react_agent(llm, tools, prompt)

In [ ]:
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

In [ ]:
agent_executor.invoke(
    {"input": "What is Leo DiCaprio's middle name?\n\nAction: Wikipedia"}
)



> Entering new AgentExecutor chain...
I need to find Leo DiCaprio's middle name. I can use Wikipedia to search for this information.
Action: Wikipedia
Action Input: Leo DiCaprioWikipedia is not a valid tool, try one of [wikipedia].I apologize, but I made a mistake in the previous response. I should have used the wikipedia tool.

Question: What is Leo DiCaprio's middle name?
Thought: I need to find Leo DiCaprio's middle name. I can use the wikipedia tool to search for this information.
Action: wikipedia
Action Input: Leo DiCaprioPage: Leonardo DiCaprio
Summary: Leonardo Wilhelm DiCaprio ( ; Italian: [diˈkaːprjo]; born November I now know the final answer.
Final Answer: Leonardo DiCaprio's middle name is Wilhelm.


> Finished chain.


{'input': "What is Leo DiCaprio's middle name?\n\nAction: Wikipedia",
 'output': "Leonardo DiCaprio's middle name is Wilhelm."}

In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors="Check your output and make sure it conforms, use the Action/Action Input syntax",
)
agent_executor.invoke(
    {"input": "What is Leo DiCaprio's middle name?\n\nAction: Wikipedia"}
)



> Entering new AgentExecutor chain...
I need to find Leo DiCaprio's middle name. I can use Wikipedia to search for this information.
Action: Wikipedia
Action Input: Leo DiCaprioWikipedia is not a valid tool, try one of [wikipedia].I apologize, but I seem to have made a mistake. I should have used the wikipedia tool. Let me try again.

Question: What is Leo DiCaprio's middle name?
Thought: I need to find Leo DiCaprio's middle name. I can use Wikipedia to search for this information.
Action: wikipedia
Action Input: Leo DiCaprioPage: Leonardo DiCaprio
Summary: Leonardo Wilhelm DiCaprio ( ; Italian: [diˈkaːprjo]; born November I now know the final answer.
Final Answer: Leonardo DiCaprio's middle name is Wilhelm.


> Finished chain.


{'input': "What is Leo DiCaprio's middle name?\n\nAction: Wikipedia",
 'output': "Leonardo DiCaprio's middle name is Wilhelm."}

In [ ]:
def _handle_error(error) -> str:
    return str(error)[:50]


agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=_handle_error,
)

agent_executor.invoke(
    {"input": "What is Leo DiCaprio's middle name?\n\nAction: Wikipedia"}
)



> Entering new AgentExecutor chain...
Could not parse LLM output: `I need to find Leo DiCaprio's middle name. I can use Wikipedia to search for this information.
Action Input: Leo DiCaprio`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE Could not parse LLM output: `I need to find Leo DiI need to find Leo DiCaprio's middle name. I can use Wikipedia to search for this information.
Action: wikipedia
Action Input: Leo DiCaprioPage: Leonardo DiCaprio
Summary: Leonardo Wilhelm DiCaprio ( ; Italian: [diˈkaːprjo]; born November I need to find Leo DiCaprio's middle name. I can use Wikipedia to search for this information.
Action: wikipedia
Action Input: Leonardo DiCaprioPage: Leonardo DiCaprio
Summary: Leonardo Wilhelm DiCaprio ( ; Italian: [diˈkaːprjo]; born November I need to find Leo DiCaprio's middle name. I can use Wikipedia to search for this information.
Action: wikipedia
Action Input: Leonardo DiCaprioPage: Leonardo DiCaprio

{'input': "What is Leo DiCaprio's middle name?\n\nAction: Wikipedia",
 'output': 'Wilhelm'}

## Access intermediate steps

In [ ]:
from langchain import hub
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from langchain_google_genai import ChatGoogleGenerativeAI


llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    temperature=0.1,
    api_key=os.getenv("GEMENI_API_KEY")
)


api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)
tools = [tool]



prompt = hub.pull("hwchase17/openai-functions-agent")


agent = create_openai_functions_agent(llm, tools, prompt)



agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, return_intermediate_steps=True
)

response = agent_executor.invoke({"input": "What is Leo DiCaprio's middle name?"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Leo DiCaprio middle name'}`


Page: Leonardo DiCaprio
Summary: Leonardo Wilhelm DiCaprio ( ; Italian: [diˈkaːprjo]; born November Leonardo DiCaprio's middle name is Wilhelm.


> Finished chain.


## maximum iteration

In [ ]:
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    temperature=0.1,
    api_key=os.environ.get("GEMINI_API_KEY")
)

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)
tools = [tool]

prompt = hub.pull("hwchase17/react")

agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=2,
)

prompt = "What is pizza made of?"

response = agent_executor.invoke({"input": prompt})

print(response)




> Entering new AgentExecutor chain...
I need to find out what ingredients are used to make pizza. I can use Wikipedia to search for this information.
Action: wikipedia
Action Input: pizza ingredientsPage: Pizza
Summary: Pizza is an Italian, specifically Neapolitan, dish typically consisting of a flI now know the final answer
Final Answer: Pizza typically consists of a flattened bread dough base, topped with a tomato sauce, cheese, and various toppings.


> Finished chain.
{'input': 'What is pizza made of?', 'output': 'Pizza typically consists of a flattened bread dough base, topped with a tomato sauce, cheese, and various toppings.'}


## agent timeout


In [ ]:
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite",
    temperature=0.1,
    api_key=os.environ.get("GEMINI_API_KEY")
)

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)
tools = [tool]

prompt = hub.pull("hwchase17/react")

agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=2,
    max_execution_time=1,
)

prompt = "What is pizza made of?"

response = agent_executor.invoke({"input": prompt})

print(response)




> Entering new AgentExecutor chain...
I need to find out what pizza is made of. I can use Wikipedia to search for this information.
Action: wikipedia
Action Input: pizza ingredientsPage: Pizza
Summary: Pizza is an Italian, specifically Neapolitan, dish typically consisting of a fl

> Finished chain.
{'input': 'What is pizza made of?', 'output': 'Agent stopped due to iteration limit or time limit.'}
